# Asignación de Variedades a Anclas — ruteo por similitud + validación robusta

| | |
|---|---|
| **Proyecto** | `ml_random_forest / ml_training` |
| **Propósito** | Decidir qué variedades entrenan modelo propio (**anclas**) y a cuál ancla rutear cada variedad con poca data |
| **Entrada** | `training/DB-HISTORICA.xlsx` (una hoja por variedad; columnas crudas se mapean a las 5 features) |
| **Salida** | `decision_table.csv/.html` (tablero) · **`variety_predictive_routing.csv/.yaml`** (el mapping accionable) |
| **Alcance** | Experimento — **no toca `src/`, `api/` ni `ui/`**. Produce el mapping `variedad → ancla` para enchufar al pipeline real |

---

## ¿Por qué existe este notebook?

Hay ~32 variedades de arándano, pero **8-9 concentran ~90% de la data** y el resto se
reparte el 10%. Con tan poca data, una variedad chica **no** puede entrenar un modelo
propio confiable. La decisión: entrenar **~11 anclas** (las de más data) y **rutear** las
demás dentro de esas anclas para que hereden un modelo entrenado con suficiente data.

> **Importante — esto NO es clustering.** El barrido de k y el silhouette muestran que
> la data **no tiene estructura de clusters natural** (silhouette negativo a todo k; los
> clusters reales están en k=2-3, no en 11). Por eso esto es **"asignar al donante más
> cercano"**, no clustering. Forzar 11 clusters "verdes" sería estadísticamente falso.

### Decisión que responde

> ¿Cuántos modelos entreno (anclas) y a cuál ancla mando cada variedad chica, de forma
> que el **pronóstico** sea lo más confiable posible?

---

## Método y por qué es estadísticamente robusto

| # | Sección | Técnica | Por qué |
|---|---------|---------|---------|
| 1 | Cargar datos | Mapeo de columnas crudas → 5 features | El Excel trae nombres crudos; corre sobre la data actual |
| 2 | Asignación | **Wasserstein + Cliff's delta** (escalado RobustScaler + filtro de outliers) | Asigna por el **mismo criterio con que decide** (efecto), no por una métrica y juzga por otra |
| 3 | Validación | **Mann-Whitney** (informativo) | Se reporta, pero **no decide** — a n grande el p-valor satura |
| 5 | Estructura global | 4 tests + **silhouette sobre todas las obs** | Veredicto honesto: Ratio/PERMANOVA son circulares y Kruskal-Wallis trivial a n grande; solo el silhouette no es circular |
| 5.05 | Barrido de k | KMeans + silhouette | Confirma que el k óptimo es 2-3, no 11 → no hay clusters |
| 5.1 | Refuerzo | **Bootstrap stability + Holm-Bonferroni** | Robustez por variedad + corrige comparaciones múltiples |
| 6 | Decisión | **Tamaño de efecto (Cliff's delta) primario**, estabilidad secundaria | No usa p-valores; un empate entre dos anclas cercanas no penaliza |
| **7** | **Ruteo predictivo** | **MAPE OOS de cada modelo de ancla sobre la variedad** | **Lo que de verdad reduce variabilidad**: rutea al ancla que mejor la *pronostica*, no a la más parecida |

> **Por qué tamaño de efecto y no p-valores:** con miles de filas, cualquier test sale
> significativo (trampa de gran muestra). Cliff's delta mide *cuánto* difieren las
> distribuciones (efecto), que es lo accionable. |d|<0.33 pequeño · <0.474 mediano.

> **Por qué ruteo predictivo (sección 7):** dos variedades pueden tener distribuciones
> distintas pero la misma relación features→target. Lo único que importa para compartir
> modelo es si el modelo del ancla **predice** bien la variedad — se mide por MAPE, no por
> distancia. Por eso la sección 7 es la decisión final; las 2-6 son el *prior*.

---

## Resultado actual (sobre la data de hoy)

- **23 variedades** en el Excel · **11 anclas** recomendadas (criterio predictivo:
  se autopredicen mejor que cualquier donante) · **12 ruteadas**.
- Veredicto de estructura: **REVISAR** (silhouette negativo) → correcto, no es clustering.
- Decisión por efecto: ~**87% confiable** (anclas + robusto/aceptable); las pocas
  "distintas" (efecto grande + poca data) se marcan honestamente para revisión.

## Cómo se conecta con el entrenamiento

Este notebook **no entrena el modelo de producción**. Genera el mapping
`variedad → ancla` (`variety_predictive_routing.yaml`) que el pipeline real (`src/` +
`task train`) puede consumir para saber qué variedades entrenan propio y cuáles heredan.


---
## ⚠️ Supuestos y límites — leer antes de accionar el mapping

Este notebook produce una **recomendación** de ruteo, no una verdad cerrada. Antes de
congelar el mapping, tener presente:

| Supuesto / límite | Por qué importa | Detalle |
|---|---|---|
| **Features proxy (sin lags)** | El ruteo se calcula sin las *lag features* del pipeline real; el **ranking** de anclas podría cambiar con las features de producción | No verificado aún |
| **Baseline contaminado por ATLAS** | ATLAS se autopronostica con MAPE ~75% e **infla el baseline** (~14.5% vs ~8.4% sin ella) → todos los `ratio` se ven mejores de lo que son | §6, [SUSTENTO §7.1](SUSTENTO_ESTADISTICO_RUTEO.md) |
| **Baseline dentro de muestra** | El MAPE del ancla es in-sample (optimista); el de la variedad es OOS (honesto) → el `ratio` compara peras con manzanas | [SUSTENTO §7.2](SUSTENTO_ESTADISTICO_RUTEO.md) |
| **Ganancia negativa = candidata a ancla** | Varias ruteadas (BELLA, ARANA, RAYMI) pronostican mejor con modelo propio | §6, [SUSTENTO §7.4](SUSTENTO_ESTADISTICO_RUTEO.md) |
| **MAPE sobre pocas filas** | Variedades con n≈35-60 deciden su ruteo con estimaciones ruidosas | — |

> **Sustento estadístico completo (para gerencia):**
> [SUSTENTO_ESTADISTICO_RUTEO.md](SUSTENTO_ESTADISTICO_RUTEO.md) — fortalezas, límites
> y recomendaciones priorizadas.

---
## Nota metodológica — ¿clustering o "otra cosa"? (leer primero)

> **Antes este notebook se planteaba como *clustering*. Ya no lo es — y esa distinción
> es el corazón de la metodología.** Este recuadro lo deja explícito.

### Dos problemas estadísticos distintos

| | **Clustering** (no supervisado) | **Lo que hacemos: ruteo a donante** (transferencia) |
|---|---|---|
| Pregunta | ¿Existen grupos *naturales* de variedades? | ¿Qué modelo-ancla **predice** mejor a cada variedad de poca data? |
| Etiqueta | No hay; se descubre | Sí: cada variedad ya tiene su `KG/JR_H` (target) |
| Validación correcta | Silhouette, gap, estabilidad Jaccard | **Error predictivo out-of-sample** (MAPE) |
| Métrica de similitud | Distancia entre distribuciones (Wasserstein) | Relación *features → target* (¿el modelo de A predice a B?) |

**Por qué importa:** dos variedades pueden tener **distribuciones distintas** y aun así
la **misma relación features→target** — en cuyo caso comparten modelo sin problema. La
distancia distribucional (clustering) responde la pregunta equivocada; lo que decide es
si el modelo del ancla **pronostica** bien la variedad.

### Diagnóstico de estructura (por qué descartamos el clustering)

Corrimos la batería estándar de validez de clusters sobre las ~16.000 observaciones:

- **Silhouette (KMeans libre):** positivo solo en **k=2–3** (~+0.33); con las 11 anclas es
  **negativo** → 11 "clusters" sobre-particionan un continuo.
- **Estabilidad Jaccard (bootstrap, Hennig):** k=2–3 → **0.96** (muy estable); pero son
  2–3 macro-regímenes, no 11 grupos.
- **HDBSCAN (densidad):** 100% ruido → **no hay clusters discretos**, es un continuo de
  productividad.

**Conclusión:** la estructura de clusters confiable existe en k=2–3, **no** en 11. Por eso
las secciones 2–6 (basadas en distribución) son un **prior diagnóstico**, no la decisión.
La decisión accionable es el **ruteo predictivo (sección 7)**.

### Principios estadísticos que seguimos (nivel experto)

1. **Tamaño de efecto > p-valor.** Con n≈10³–10⁴ todo test es significativo (trampa de
   gran muestra). Usamos **Cliff's delta** (no paramétrico, robusto) y umbrales de efecto,
   no significancia. El p-valor de Mann-Whitney queda como *informativo*.
2. **Evitar circularidad.** Ratio intra/inter y PERMANOVA usan como etiqueta el ancla
   *más cercana por la misma distancia* → casi tautológicos. No votan el veredicto.
3. **Corrección por comparaciones múltiples.** 5 features ⇒ FWER ≈ 1−0.95⁵ ≈ 23%;
   **Holm-Bonferroni** lo controla al 5%.
4. **Robustez por remuestreo.** **Bootstrap** convierte cada asignación de etiqueta a
   probabilidad y, vía binomial vs azar (1/k) + Holm, a significancia.
5. **Pre-procesamiento consistente.** `RobustScaler` (mediana/IQR) + filtro de outliers
   (IsolationForest); la **validación corre sobre la misma representación** que la asignación.
6. **La decisión final es predictiva.** Lo único que reduce la varianza del pronóstico es
   el **MAPE out-of-sample** del modelo donante sobre la variedad (sección 7).


In [1]:
# ══════════════════════════════════════════════════════════════════
# Setup — la lógica vive en variety_routing.py (importable y testeable);
# el notebook se queda con orquestación + visualización.
# ══════════════════════════════════════════════════════════════════
import os
import sys
import warnings

sys.path.insert(0, os.path.abspath("."))  # para importar variety_routing.py
sys.path.insert(0, os.path.abspath("../src"))
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from variety_routing import (
    EFFECT_MEDIUM,
    EFFECT_SMALL,
    MIN_N_VALIDATE,
    RANDOM_STATE,
    VIABILITY_ZONES,
    ExperimentConfig,
    apply_holm,
    assign_to_anchors,
    bootstrap_stability,
    build_final_result,
    classify_viability,
    cliffs_delta,
    compute_group_summary,
    route_to_anchors_predictive,
    filter_outliers,
    load_variety_data,
    mann_whitney_holm,
    mean_wasserstein,
    normalize_features,
    run_all_validations,
    silhouette_over_observations,
    validate_effect_size,
    validate_mann_whitney,
)

sns.set_style("whitegrid")
np.random.seed(RANDOM_STATE)


# ══════════════════════════════════════════════════════════════════
# VISUALIZACIÓN (específica del notebook)
# ══════════════════════════════════════════════════════════════════
THRESHOLD_LINES = [
    {"key": "robust", "color": "#2ecc71", "label": "Robusto"},
    {"key": "moderate", "color": "#f39c12", "label": "Básico"},
    {"key": "conservative", "color": "#e74c3c", "label": "Mínimo absoluto"},
]


def _add_threshold_lines(ax, thresholds: dict):
    """Agrega las líneas de umbral de viabilidad al gráfico."""
    for tl in THRESHOLD_LINES:
        ax.axhline(
            y=thresholds[tl["key"]],
            color=tl["color"],
            linestyle="--",
            linewidth=2,
            label=f"{tl['label']} (≥{thresholds[tl['key']]})",
        )


def plot_viability_chart(summary_df: pd.DataFrame, thresholds: dict):
    """Barras: clasificación de variedades por viabilidad de entrenamiento."""
    df = summary_df.sort_values("n_filas", ascending=False)
    colors = [classify_viability(r["n_filas"], thresholds)["color"] for _, r in df.iterrows()]

    fig, ax = plt.subplots(figsize=(16, 7))
    ax.bar(range(len(df)), df["n_filas"], color=colors)
    ax.set_xticks(range(len(df)))
    ax.set_xticklabels(df["variedad"], rotation=90, fontsize=8)
    _add_threshold_lines(ax, thresholds)
    ax.set_ylabel("Número de filas")
    ax.set_title("Clasificación de variedades por viabilidad de entrenamiento LightGBM")
    ax.legend(loc="upper right")
    plt.tight_layout()
    plt.show()

    bins = {
        "✓ Robusto": (df["n_filas"] >= thresholds["robust"]).sum(),
        "~ Básico": ((df["n_filas"] >= thresholds["moderate"]) & (df["n_filas"] < thresholds["robust"])).sum(),
        "⚠ Mínimo": ((df["n_filas"] >= thresholds["conservative"]) & (df["n_filas"] < thresholds["moderate"])).sum(),
        "✗ Insuficiente": (df["n_filas"] < thresholds["conservative"]).sum(),
    }
    for label, count in bins.items():
        print(f"{label}: {count}")


def plot_group_composition(group_summary: pd.DataFrame):
    """Barras apiladas: composición (propias + asignadas) de cada grupo ancla."""
    fig, ax = plt.subplots(figsize=(18, 8))
    x = range(len(group_summary))
    ax.bar(x, group_summary["filas_propias"], label="Filas propias (ancla)",
           color="#2ecc71", edgecolor="black", linewidth=0.5)
    ax.bar(x, group_summary["filas_asignadas"], bottom=group_summary["filas_propias"].values,
           label="Filas de variedades asignadas", color="#3498db", edgecolor="black", linewidth=0.5)
    ax.set_xticks(list(x))
    ax.set_xticklabels(group_summary["ancla"], rotation=45, ha="right", fontsize=10)
    for i, (_, row) in enumerate(group_summary.iterrows()):
        ax.text(i, row["total"] + 15, f"+{row['n_asignadas']} vars\n({row['total']} total)",
                ha="center", va="bottom", fontsize=8, fontweight="bold")
    ax.set_ylabel("Número de filas")
    ax.set_title("Composición de grupos de entrenamiento por ancla")
    ax.legend(loc="upper right", fontsize=10)
    plt.tight_layout()
    plt.show()


def plot_assignment_chart(final_result: pd.DataFrame, thresholds: dict):
    """Barras estilo viabilidad con el ancla asignada anotada sobre cada barra."""
    plot_df = final_result.sort_values("n_filas", ascending=False).reset_index(drop=True)
    colors = [classify_viability(r["n_filas"], thresholds)["color"] for _, r in plot_df.iterrows()]
    labels = [
        f"⚓ {r['variedad']}" if r["variedad"] == r["entrena_con"] else r["variedad"]
        for _, r in plot_df.iterrows()
    ]

    fig, ax = plt.subplots(figsize=(20, 8))
    ax.bar(range(len(plot_df)), plot_df["n_filas"], color=colors)
    ax.set_xticks(range(len(plot_df)))
    ax.set_xticklabels(labels, rotation=90, fontsize=8)
    _add_threshold_lines(ax, thresholds)
    for i, (_, row) in enumerate(plot_df.iterrows()):
        if row["variedad"] != row["entrena_con"]:
            ax.annotate(
                f"→ {row['entrena_con']}",
                xy=(i, row["n_filas"] + 5), ha="center", va="bottom",
                fontsize=6.5, fontweight="bold", color="#2c3e50", rotation=70,
                bbox=dict(boxstyle="round,pad=0.15", facecolor="white",
                          edgecolor="gray", alpha=0.85, linewidth=0.5),
            )
    ax.set_ylabel("Número de filas")
    ax.set_title("Asignación de variedades a anclas — por viabilidad de entrenamiento LightGBM")
    ax.legend(loc="upper right", fontsize=9)
    plt.tight_layout()
    plt.show()


# ── Inicializar config ──
config = ExperimentConfig()
print(f"Anclas: {len(config.anchor_varieties)} | Features: {config.features}")
print(f"Umbrales: {config.thresholds}")
print("Setup OK ✓")


Anclas: 11 | Features: ['KGHECT', 'INDUSTRIAL', 'DPC', 'PesoBayaFIMPRO', 'KGHORA']
Umbrales: {'conservative': 80, 'moderate': 160, 'robust': 638}
Setup OK ✓


---
## 1. Cargar datos

Leemos `training/DB-HISTORICA.xlsx` — una hoja por variedad. La función
`load_variety_data()` filtra las 5 features que interesan (definidas en
`config.features`) y descarta filas con valores nulos en cualquiera de ellas.

**Separación en dos grupos:**

- **Anclas** (11): variedades que entrenarán su propio modelo. Están fijadas por negocio en `config.anchor_varieties`.
- **No-anclas** (12): el resto. Se asigna un ancla (sección 2) y se rutea por predicción (sección 7).

> **Qué esperar:** ~16.000 filas entre las ~23 variedades. Variedades con
> muy pocas filas (n<20) se marcan `insuficiente` — no se validan de forma fiable
> y se documentan en la tabla de decisión final.


In [2]:
all_data, anchor_names, non_anchor_names, summary_df = load_variety_data(config)

print(f"Total variedades: {len(all_data)}")
print(f"\nAnclas ({len(anchor_names)}):")
for a in anchor_names:
    print(f"  ⚓ {a:20s} ({len(all_data[a]):4d} filas)")

print(f"\nA asignar ({len(non_anchor_names)}):")
for v in non_anchor_names:
    print(f"    {v:20s} ({len(all_data[v]):4d} filas)")

summary_df

Total variedades: 23

Anclas (11):
  ⚓ POP                  (9990 filas)
  ⚓ VENTURA              (4656 filas)
  ⚓ BEAUTY               (4125 filas)
  ⚓ BIANCA               (3347 filas)
  ⚓ ATLAS                (2755 filas)
  ⚓ JUPITER              (1668 filas)
  ⚓ ROSITA               ( 588 filas)
  ⚓ BELLA                ( 664 filas)
  ⚓ ARANA                ( 546 filas)
  ⚓ EMERALD              ( 803 filas)
  ⚓ MAGICA               (1152 filas)

A asignar (12):
    AZRA                 ( 131 filas)
    BILOXI               ( 400 filas)
    BONITA               ( 235 filas)
    FCM17-132            ( 116 filas)
    KIRRA                ( 844 filas)
    MADEIRA              ( 167 filas)
    MAGNUS               ( 132 filas)
    MALIBU               ( 230 filas)
    MASIRAH              ( 188 filas)
    RAYMI                ( 319 filas)
    STELLA               ( 151 filas)
    TERRAPIN             ( 105 filas)


,variedad,n_filas,tipo
9,POP,9990,⚓ ANCLA
0,VENTURA,4656,⚓ ANCLA
15,BEAUTY,4125,⚓ ANCLA
8,BIANCA,3347,⚓ ANCLA
4,ATLAS,2755,⚓ ANCLA
1,JUPITER,1668,⚓ ANCLA
7,MAGICA,1152,⚓ ANCLA
13,KIRRA,844,a asignar
6,EMERALD,803,⚓ ANCLA
18,BELLA,664,⚓ ANCLA


In [3]:
plot_viability_chart(summary_df, config.thresholds)

✓ Robusto: 10
~ Básico: 8
⚠ Mínimo: 5
✗ Insuficiente: 0


---
### ⚓ Set de anclas — fuente única de verdad *(reconciliado 2026-06-26)*

Antes el notebook usaba **dos sets de anclas distintos** sin reconciliarlos: uno en
`config.anchor_varieties` (§1, prior) y otro hardcodeado en §7 (decisión). El
diagnóstico de soporte (§2–5) validaba entonces **una partición que la decisión no
usaba**. **Corregido:** ahora hay **una sola lista** en `config.anchor_varieties`, que
consumen **§1–§5 (prior)** y **§7 (decisión)** → describen exactamente el mismo set.

**11 anclas** (criterio: se autopredicen mejor que cualquier donante, MAPE OOS §7):
`POP · BEAUTY · VENTURA · BIANCA · ATLAS · JUPITER · MAGICA · EMERALD · KIRRA · ROSITA · BILOXI`

Cambios respecto del set anterior:
- **Entran** KIRRA y BILOXI (excelentes auto-predictores: MAPE OOS 6.3% y 4.6%).
- **Salen** BELLA y ARANA → pasan a ruteadas. *Pero* tienen `n` holgado (195 y 311) y
  **ganancia negativa** (§6): son **candidatas a re-promover a ancla**. Decisión
  abierta — ver [SUSTENTO_ESTADISTICO_RUTEO.md](SUSTENTO_ESTADISTICO_RUTEO.md) §7.4/§8.

> ⚠️ **Re-ejecutar el notebook** tras este cambio: §2–§5 (prior) se recalculan sobre el
> nuevo set; §7 (decisión) ya usaba este set, así que el mapping de salida no cambia.

---
## 2. Distancia de Wasserstein y asignación a anclas

Para cada variedad no-ancla medimos qué tan parecida es su **distribución** a cada
una de las anclas. La **asignación** se hace por **tamaño de efecto** (Cliff's delta),
con Wasserstein como desempate y como métrica reportada — así se asigna con el mismo
criterio con que luego se valida (§5.1), no por una métrica y se juzga por otra.

### Cómo funciona (en orden)

1. **Normalización:** `RobustScaler` sobre las 5 features (centra en la mediana, escala por el IQR) — resistente a outliers, a diferencia de `StandardScaler`.
2. **Distancia por feature:** para cada feature calculamos la distancia Wasserstein-1 entre la variedad y el ancla.
3. **Distancia agregada:** promedio de las 5 distancias por feature (columna `distancia_wass`, reportada).
4. **Asignación:** la variedad va al ancla con **menor `max|Cliff's delta|`** (tamaño de efecto); Wasserstein desempata. *No* se asigna por la distancia promedio sola.
5. **Validación robusta:** la similitud variedad↔ancla se juzga por **tamaño de efecto (Cliff's delta)** sobre la misma data escalada+filtrada, no por p-valores (que saturan a n grande). Mann-Whitney queda como informativo.

### ¿Por qué Wasserstein y no otra métrica?

| Métrica | ¿Qué mide? | Problema para este caso |
|---------|-----------|-------------------------|
| **Euclidiana** | Distancia entre medias | Ignora forma y varianza — 2 variedades con misma media pero muy distintas en varianza se verían iguales |
| **KL divergence** | Diferencia entre densidades | Requiere estimar densidades (KDE), asume continuidad, explota cuando hay poco solapamiento |
| **Wasserstein** ✓ | "Costo" de transformar una distribución en otra | Captura forma completa; robusta con pocos datos; interpretable en las unidades originales |

### Qué salida produce

`assignment_df` — una fila por variedad no-ancla con: `variedad`, `ancla_asignada`, `distancia_wass`, y las distancias top-2 para entender si la asignación fue "obvia" o "ajustada".

In [4]:
scaled_data = normalize_features(all_data, config.features)
scaled_data = filter_outliers(
    scaled_data,
    config.features,
    contamination=config.outlier_contamination,
    random_state=RANDOM_STATE,
)
print(
    f"Outliers filtrados "
    f"(contamination={config.outlier_contamination:.0%} por variedad)"
)
assignment_df = assign_to_anchors(
    non_anchor_names, anchor_names, all_data, scaled_data, config.features
)

# Mostrar agrupado por ancla
print("=" * 80)
print("ASIGNACIÓN DE VARIEDADES A ANCLAS (Wasserstein)")
print("=" * 80)

for anchor in sorted(anchor_names):
    assigned = assignment_df[assignment_df["ancla_asignada"] == anchor]
    total_group = len(all_data[anchor]) + assigned["n_filas"].sum()
    print(f"\n{'─' * 70}")
    print(f"  ⚓ {anchor} ({len(all_data[anchor])} propias, {total_group} total)")
    print(f"{'─' * 70}")
    if len(assigned) > 0:
        for _, row in assigned.sort_values("distancia_wass").iterrows():
            print(
                f"    ← {row['variedad']:20s} ({row['n_filas']:4d} filas) "
                f"dist={row['distancia_wass']:.3f}  "
                f"(2da: {row['segunda_opcion']} dist={row['dist_segunda']:.3f})"
            )
    else:
        print("    (sin variedades asignadas)")

print(f"\nTotal asignaciones: {len(assignment_df)}")
assignment_df

Outliers filtrados (contamination=5% por variedad)


ASIGNACIÓN DE VARIEDADES A ANCLAS (Wasserstein)

──────────────────────────────────────────────────────────────────────
  ⚓ ARANA (546 propias, 865 total)
──────────────────────────────────────────────────────────────────────
    ← RAYMI                ( 319 filas) dist=0.228  (2da: ROSITA dist=0.334)

──────────────────────────────────────────────────────────────────────
  ⚓ ATLAS (2755 propias, 3094 total)
──────────────────────────────────────────────────────────────────────
    ← MASIRAH              ( 188 filas) dist=0.240  (2da: BIANCA dist=0.230)
    ← STELLA               ( 151 filas) dist=0.340  (2da: JUPITER dist=0.175)

──────────────────────────────────────────────────────────────────────
  ⚓ BEAUTY (4125 propias, 4125 total)
──────────────────────────────────────────────────────────────────────
    (sin variedades asignadas)

──────────────────────────────────────────────────────────────────────
  ⚓ BELLA (664 propias, 795 total)
───────────────────────────────────────────

,variedad,n_filas,ancla_asignada,distancia_wass,segunda_opcion,dist_segunda
9,RAYMI,319,ARANA,0.2276,ROSITA,0.3335
8,MASIRAH,188,ATLAS,0.2405,BIANCA,0.2297
10,STELLA,151,ATLAS,0.3403,JUPITER,0.1747
0,AZRA,131,BELLA,0.3423,ROSITA,0.3092
11,TERRAPIN,105,BIANCA,0.2330,VENTURA,0.2586
2,BONITA,235,JUPITER,0.1622,BIANCA,0.2442
4,KIRRA,844,JUPITER,0.1525,BIANCA,0.1669
3,FCM17-132,116,POP,0.6410,BEAUTY,0.5967
6,MAGNUS,132,ROSITA,0.3443,BIANCA,0.3729
1,BILOXI,400,VENTURA,0.4048,JUPITER,0.2612


---
## 3. Validación — Mann-Whitney U *(informativo, NO decide)*

> ⚠️ **Esta sección no toma ninguna decisión.** Con miles de filas el p-valor
> satura (*trampa de la gran muestra*): casi todo sale "significativamente distinto".
> Por eso Mann-Whitney se reporta como **lectura informativa**; la **asignación** la
> fija el tamaño de efecto (Cliff's delta, §2/§5.1) y la **decisión final** el error
> de pronóstico (MAPE OOS, §6).

Para cada asignación variedad → ancla aplicamos Mann-Whitney U feature por feature y
contamos cuántas tienen `p > 0.05` (no se distinguen → "similares").

### Cómo leerlo (orientativo, no es un umbral de decisión)

| % features similares | Lectura |
|---|---|
| **≥ 80%** | La variedad se ve muy parecida al ancla |
| **60–79%** | Parecida |
| **< 60%** | Difiere en varias features — *no* implica que herede mal; eso lo dice el **MAPE (§6)** |

> **Por qué NO descartamos las "débiles":** aunque los tests univariados salgan
> significativos, lo único que importa para compartir modelo es si el ancla
> **pronostica** bien la variedad (§6). La §5.1 aplica además **Holm-Bonferroni**
> para corregir el inflado del error tipo I de correr 5 tests.

> **Edge case conocido:** variedades con `n < 3` filas (ej. histórico `JULIETA`)
> devuelven 0% porque Mann-Whitney requiere ≥3 observaciones; la tabla final las marca
> como `"insuficiente"`, no `"revisar"`.

In [5]:
validation_df = validate_mann_whitney(
    assignment_df, all_data, config.features, config.alpha
)

print("=" * 80)
print("VALIDACIÓN: Mann-Whitney U (p > 0.05 = similares)")
print("=" * 80)
print(f"\n  {'variedad':20s}   {'ancla':20s}  {'similares':>10s} {'%':>6s}")
print("  " + "─" * 68)
for _, r in validation_df.iterrows():
    print(
        f"  {r['variedad']:20s} → {r['ancla']:20s} "
        f"{r['features_similares']:>10s} {r['pct_similar']:5.0f}%  {r['status']}"
    )

good = (validation_df["pct_similar"] >= config.min_similar_pct).sum()
print(
    f"\n✓ {good}/{len(validation_df)} asignaciones con ≥{config.min_similar_pct:.0f}% features similares"
)

low = validation_df[validation_df["pct_similar"] < config.min_similar_pct]
if len(low) > 0:
    print(
        f"\n⚠ Asignaciones con <{config.min_similar_pct:.0f}% similitud ({len(low)}):"
    )
    for _, r in low.iterrows():
        print(f"    {r['variedad']} → {r['ancla']} ({r['pct_similar']:.0f}%)")
    print("    (se asignan igualmente al ancla más cercana por Wasserstein)")


# ── Validación robusta por tamaño de efecto (decisión real) ──────────
effect_df = validate_effect_size(assignment_df, scaled_data, config.features)
print()
print("=" * 80)
print("VALIDACIÓN ROBUSTA: tamaño de efecto (Cliff's delta, decide la asignación)")
print("=" * 80)
print(f"  {'variedad':20s}   {'ancla':14s} {'n':>5s} {'max|Cliff|':>11s}  veredicto")
print("  " + "─" * 70)
for _, r in effect_df.iterrows():
    mc = f"{r['max_cliff']:.2f}" if pd.notna(r["max_cliff"]) else "  —"
    print(f"  {r['variedad']:20s} → {r['ancla']:14s} {int(r['n']):5d} {mc:>11s}  {r['veredicto']}")
n_similar = (effect_df["veredicto"] == "similar").sum()
print(
    f"\n✓ {n_similar}/{len(effect_df)} asignaciones 'similar' por tamaño de efecto "
    f"(max|Cliff's delta| < {EFFECT_SMALL})"
)

VALIDACIÓN: Mann-Whitney U (p > 0.05 = similares)

  variedad               ancla                  similares      %
  ────────────────────────────────────────────────────────────────────
  AZRA                 → BELLA                       0/5     0%  ⚠
  STELLA               → ATLAS                       0/5     0%  ⚠
  MADEIRA              → VENTURA                     0/5     0%  ⚠
  KIRRA                → JUPITER                     1/5    20%  ⚠
  BILOXI               → VENTURA                     1/5    20%  ⚠
  TERRAPIN             → BIANCA                      2/5    40%  ⚠
  BONITA               → JUPITER                     2/5    40%  ⚠
  RAYMI                → ARANA                       2/5    40%  ⚠
  MALIBU               → VENTURA                     2/5    40%  ⚠
  FCM17-132            → POP                         2/5    40%  ⚠
  MASIRAH              → ATLAS                       3/5    60%  ✓
  MAGNUS               → ROSITA                      3/5    60%  ✓

✓ 2/12 a

---
## 4. Resultado final

Combinamos las anclas (con sus propios datos) y las no-anclas (con su ancla asignada)
en un único `final_result`, que alimenta la tabla de decisión (sección 6) y el ruteo
predictivo (sección 7).

> El artefacto accionable lo produce la **sección 7** (`variety_predictive_routing.yaml`),
> no esta sección. Aquí solo consolidamos el resultado intermedio.


In [6]:
final_result = build_final_result(
    anchor_names, assignment_df, validation_df, all_data, config.thresholds
)
group_summary = compute_group_summary(anchor_names, assignment_df, all_data)

# Gráfico 1: Composición por grupo
plot_group_composition(group_summary)

# Gráfico 2: Asignación estilo viabilidad
plot_assignment_chart(final_result, config.thresholds)

# Resumen
print(f"Total filas para entrenamiento: {group_summary['total'].sum()}")
print(f"Anclas: {len(anchor_names)} | Asignadas: {len(assignment_df)}")
final_result

Total filas para entrenamiento: 33312
Anclas: 11 | Asignadas: 12


,variedad,n_filas,zona,entrena_con,ancla_asignada,distancia_wass,nota
11,RAYMI,319,⬛ Agrupar,ARANA,ARANA,0.2276,"Vecino más cercano: ARANA (dist=0.228, similit..."
0,ARANA,546,🟡 Individual,ARANA,ARANA,0.0000,Modelo individual (árboles básicos)
12,MASIRAH,188,⬛ Agrupar,ATLAS,ATLAS,0.2405,"Vecino más cercano: ATLAS (dist=0.240, similit..."
13,STELLA,151,⬛ Agrupar,ATLAS,ATLAS,0.3403,"Vecino más cercano: ATLAS (dist=0.340, similit..."
1,ATLAS,2755,🟢 Individual,ATLAS,ATLAS,0.0000,Modelo individual con nested CV completo
2,BEAUTY,4125,🟢 Individual,BEAUTY,BEAUTY,0.0000,Modelo individual con nested CV completo
14,AZRA,131,⬛ Agrupar,BELLA,BELLA,0.3423,"Vecino más cercano: BELLA (dist=0.342, similit..."
3,BELLA,664,🟢 Individual,BELLA,BELLA,0.0000,Modelo individual con nested CV completo
15,TERRAPIN,105,⬛ Agrupar,BIANCA,BIANCA,0.2330,"Vecino más cercano: BIANCA (dist=0.233, simili..."
4,BIANCA,3347,🟢 Individual,BIANCA,BIANCA,0.0000,Modelo individual con nested CV completo


In [7]:
# El resultado final no se exporta a CSV aquí (los artefactos accionables son
# decision_table.* en la sección 6 y variety_predictive_routing.* en la sección 7).
final_result


,variedad,n_filas,zona,entrena_con,ancla_asignada,distancia_wass,nota
11,RAYMI,319,⬛ Agrupar,ARANA,ARANA,0.2276,"Vecino más cercano: ARANA (dist=0.228, similit..."
0,ARANA,546,🟡 Individual,ARANA,ARANA,0.0000,Modelo individual (árboles básicos)
12,MASIRAH,188,⬛ Agrupar,ATLAS,ATLAS,0.2405,"Vecino más cercano: ATLAS (dist=0.240, similit..."
13,STELLA,151,⬛ Agrupar,ATLAS,ATLAS,0.3403,"Vecino más cercano: ATLAS (dist=0.340, similit..."
1,ATLAS,2755,🟢 Individual,ATLAS,ATLAS,0.0000,Modelo individual con nested CV completo
2,BEAUTY,4125,🟢 Individual,BEAUTY,BEAUTY,0.0000,Modelo individual con nested CV completo
14,AZRA,131,⬛ Agrupar,BELLA,BELLA,0.3423,"Vecino más cercano: BELLA (dist=0.342, similit..."
3,BELLA,664,🟢 Individual,BELLA,BELLA,0.0000,Modelo individual con nested CV completo
15,TERRAPIN,105,⬛ Agrupar,BIANCA,BIANCA,0.2330,"Vecino más cercano: BIANCA (dist=0.233, simili..."
4,BIANCA,3347,🟢 Individual,BIANCA,BIANCA,0.0000,Modelo individual con nested CV completo


## 5. Diagnóstico de estructura *(informativo — NO decide)*

Corremos 4 tests clásicos de validez de estructura, pero con una advertencia que el
código aplica explícitamente:

> **Ninguno de estos 4 tests vota el veredicto.** Tres son engañosos con esta data:
> - **Ratio intra/inter** y **PERMANOVA** usan como etiqueta el ancla *más cercana
>   por la misma distancia* → son casi **circulares** (se validan a sí mismos).
> - **Kruskal-Wallis** es **trivial a n grande**: con miles de filas casi siempre sale
>   significativo.
>
> El **único juez honesto** es el **Silhouette sobre TODAS las observaciones** (§5.1):
> no es circular y tiene potencia real. Si sale **≤ 0 → no hay clusters naturales**
> (lo esperado: esto es ruteo a donante, no clustering).

| # | Test | Pregunta que responde | Rol |
|---|------|-----------------------|-----|
| 1 | **Silhouette (centroides)** | ¿Cada variedad está más cerca de su ancla que de las otras? | informativo (baja potencia: 1 punto/variedad) |
| 2 | **Ratio intra/inter** | ¿La distancia dentro del grupo es menor que entre grupos? | informativo (**circular**) |
| 3 | **Kruskal-Wallis** | ¿Los grupos difieren en cada feature? | informativo (**trivial a n grande**) |
| 4 | **PERMANOVA** | ¿La partición explica la varianza mejor que el azar? | informativo (**circular**) |

**Veredicto:** lo emite el código a partir del **silhouette sobre observaciones**, no
de un conteo "≥3 de 4 pasan". La decisión por variedad la dan el **tamaño de efecto +
estabilidad bootstrap (§5.1)** y, finalmente, el **MAPE OOS (§6)**.

In [8]:
validation_tests = run_all_validations(
    final_result,
    assignment_df,
    anchor_names,
    all_data,
    scaled_data,
    config.features,
    config.n_permutations,
)


✗ Silhouette Score
   Score global: -0.0440 — Bajo
   ⚓ ARANA               : +0.365
   ⚓ ATLAS               : -0.241
   ⚓ BELLA               : +0.212
   ⚓ BIANCA              : +0.062
   ⚓ JUPITER             : +0.094
   ⚓ POP                 : -0.261
   ⚓ ROSITA              : -0.048
   ⚓ VENTURA             : -0.275

✓ Ratio intra/inter
   Intra-grupo: 0.2982  |  Inter-grupo: 0.4734
   Ratio: 0.6299 — Bueno

✓ Kruskal-Wallis
   KGHECT                H=   1544.27  p=0.00e+00  ✓ Sig.
   INDUSTRIAL            H=   3176.84  p=0.00e+00  ✓ Sig.
   DPC                   H=  12036.17  p=0.00e+00  ✓ Sig.
   PesoBayaFIMPRO        H=   7540.87  p=0.00e+00  ✓ Sig.
   KGHORA                H=   3462.55  p=0.00e+00  ✓ Sig.
   → 5/5 features significativas

✓ PERMANOVA
   F=4.1660  p=0.0010  R²=0.7764 (77.6%)



VEREDICTO HONESTO — solo evidencia no-circular
   Informativos (NO votan): Ratio y PERMANOVA son circulares;
   Kruskal-Wallis es trivial a n grande.
   Silhouette sobre 16,047 observaciones: -0.1120

   → ❌ SIN ESTRUCTURA NATURAL — esto es 'modelo donante más cercano', no clusters. La decisión por variedad la dan bootstrap + Cliff's delta (5.1).


---
## 5.05 Validación de k óptimo (barrido silhouette)

El silhouette global de la sección 5.1 puede salir **negativo** si el número
de anclas (k) no coincide con la estructura natural de los datos. Aquí
hacemos un barrido de k = `k_sweep_min` a `k_sweep_max` usando KMeans
sobre las observaciones escaladas (ya filtradas de outliers), y comparamos
el silhouette que obtiene la asignación **inducida por las anclas actuales**
contra el mejor k que encontraría KMeans libre.

- Si el codo está en el **mismo k** que `len(anchor_varieties)`: el número
  de anclas es correcto.
- Si el mejor k es **menor**: hay anclas redundantes, considerar fusionar
  algunas.
- Si el mejor k es **mayor**: las anclas actuales no capturan toda la
  heterogeneidad; considerar agregar una ancla nueva.


In [9]:
# ══════════════════════════════════════════════════════════════════
# 5.05  Barrido de k óptimo por silhouette
# ══════════════════════════════════════════════════════════════════
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Apilar todas las observaciones escaladas (post-outlier filter)
obs_parts_k = [df[config.features].values for df in scaled_data.values()]
X_all_k = np.vstack(obs_parts_k)

k_current = len(anchor_names)
k_range = list(range(config.k_sweep_min, config.k_sweep_max + 1))
sil_scores_k = []

print("=" * 70)
print(f"Barrido k = {k_range[0]}..{k_range[-1]}  (k actual = {k_current})")
print("=" * 70)
for k in k_range:
    km = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    labels_k = km.fit_predict(X_all_k)
    sil_k = silhouette_score(X_all_k, labels_k, metric="euclidean")
    sil_scores_k.append(sil_k)
    mark = " ← k actual" if k == k_current else ""
    print(f"  k={k:>2}  silhouette={sil_k:+.4f}{mark}")

best_k = k_range[int(np.argmax(sil_scores_k))]
best_sil = max(sil_scores_k)
sil_at_current = sil_scores_k[k_range.index(k_current)] if k_current in k_range else float("nan")

print()
print(f"  Mejor k (KMeans libre):   k={best_k}  silhouette={best_sil:+.4f}")
print(f"  Silhouette con k actual:  k={k_current}  silhouette={sil_at_current:+.4f}")
delta_k = best_k - k_current
if abs(delta_k) <= 1:
    verdict_k = "✓ k actual coincide con el óptimo (±1)"
elif delta_k < 0:
    verdict_k = f"⚠ Posible redundancia: óptimo k={best_k} < actual k={k_current}"
else:
    verdict_k = f"⚠ Heterogeneidad no cubierta: óptimo k={best_k} > actual k={k_current}"
print(f"  Veredicto: {verdict_k}")

# Gráfico del codo
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(k_range, sil_scores_k, marker="o", linewidth=2)
ax.axvline(k_current, color="#f59e0b", linestyle="--", label=f"k actual = {k_current}")
ax.axvline(best_k, color="#10b981", linestyle="--", label=f"k óptimo = {best_k}")
ax.set_xlabel("k (número de clusters)")
ax.set_ylabel("Silhouette global")
ax.set_title("Barrido de k óptimo — curva codo")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


Barrido k = 3..15  (k actual = 11)


  k= 3  silhouette=+0.3253


  k= 4  silhouette=+0.2853


  k= 5  silhouette=+0.2486


  k= 6  silhouette=+0.2421


...

  k=11  silhouette=+0.2312 ← k actual


  k=12  silhouette=+0.2214


  k=13  silhouette=+0.2217


  k=14  silhouette=+0.2132


  k=15  silhouette=+0.2086

  Mejor k (KMeans libre):   k=3  silhouette=+0.3253
  Silhouette con k actual:  k=11  silhouette=+0.2312
  Veredicto: ⚠ Posible redundancia: óptimo k=3 < actual k=11


---
## 5.1 Análisis estadístico reforzado

Los 4 tests de la sección 5 dan una **foto global** (¿hay señal?), pero tienen
tres limitaciones conocidas con esta data:

| Limitación | Impacto | Corrección aplicada |
|---|---|---|
| Silhouette + PERMANOVA corren sobre **los centroides** (1 por variedad) en vez de las ~16.000 observaciones | Baja potencia estadística | **Silhouette global** sobre todas las obs |
| Mann-Whitney por feature **sin corrección** de comparaciones múltiples | α real ≈ 23% (no 5%) con 5 tests → % similares inflado | **Holm-Bonferroni** step-down |
| No sabemos qué tan **frágil** es cada asignación individual | "Revisar" es una categoría opaca | **Bootstrap 100×** por variedad |

### Qué aporta cada técnica

#### (a) Bootstrap stability — ¿es robusta la asignación?

Para cada variedad no-ancla: resampleamos con reemplazo sus filas 100 veces; en cada iteración recomputamos su ancla nearest por Wasserstein. **Stability %** = proporción de iteraciones que cayeron en el ancla canónica.

- **≥ 90%** → asignación muy robusta (la data es consistente con el ancla)
- **60–89%** → asignación razonable pero con alternativa competitiva
- **< 60%** → asignación frágil; considerar entrenarla como ancla propia o revisar datos

Este es **el número más accionable para gerencia**: pasa "revisar" de etiqueta a probabilidad.

#### (b) Silhouette sobre todas las observaciones

El Silhouette de la sección 5 usa un centroide por variedad. Con tan pocos puntos en 5D y 11 grupos, el test tiene poca potencia. Aquí lo corremos sobre las ~16.000 observaciones individuales etiquetadas por su ancla asignada — más realista y con mayor poder estadístico.

#### (c) Mann-Whitney con corrección Holm-Bonferroni

Correr MW 5 veces (una por feature) con α=0.05 infla el error tipo I: probabilidad de al menos un falso positivo ≈ **1 - 0.95⁵ = 23%**. Holm-Bonferroni es un step-down que ajusta cada umbral por el rank del p-value, más potente que Bonferroni simple y sigue controlando el FWER al 5%.

La métrica `pct_similar_holm` es una versión más honesta de `pct_similar`: típicamente algunas asignaciones "aceptables" caen al aplicar la corrección.

### Cómo leer los resultados

| Si ves… | Significa… |
|---|---|
| Variedad con `stability < 60%` y `pct_similar_holm < 40%` | Asignación realmente frágil — evaluar entrenar como ancla propia |
| Variedad con `stability ≥ 90%` y `pct_similar_holm < 60%` | Ancla nearest robusta pero feature-level difiere — aceptable para modelo de regresión |
| `sil_all` mucho mayor que `sil_centroides` | La estructura del grupo es real pero se diluye al agregar por centroide |
| `n_raw - n_holm > 3` | La corrección importa — el reporte sin corrección era demasiado optimista |


In [10]:
# ══════════════════════════════════════════════════════════════════
# 5.1  Análisis estadístico reforzado (usa variety_routing — sin duplicar)
# ------------------------------------------------------------------
#   (a) Bootstrap stability — ¿qué tan frágil es cada asignación?
#   (b) Silhouette sobre TODAS las observaciones (no solo centroides)
#   (c) Mann-Whitney U con corrección Holm-Bonferroni + umbral calibrado
#
# Produce stability_df, sil_all, mw_holm_df, mw_holm_threshold_calibrated
# que consume la sección 6 (tabla de decisión y YAML).
# ══════════════════════════════════════════════════════════════════
BOOTSTRAP_ITERATIONS = config.bootstrap_iterations
BOOTSTRAP_SEED = config.bootstrap_seed
k_anc = len(anchor_names)

# ── 5.1.1  Bootstrap stability ──────────────────────────────────────
print("=" * 70)
print(f"Bootstrap stability ({BOOTSTRAP_ITERATIONS} iter por variedad no-ancla)")
print("=" * 70)

anchor_of_map = {r["variedad"]: r["entrena_con"] for _, r in final_result.iterrows()}
non_anchor_list = [v for v in all_data if v not in anchor_names]
stability_df = bootstrap_stability(
    non_anchor_list, anchor_names, scaled_data, anchor_of_map, config.features,
    iterations=BOOTSTRAP_ITERATIONS, seed=BOOTSTRAP_SEED, alpha=config.alpha,
)

n_sig_boot = int(stability_df["boot_significant"].sum())
n_robust = int((stability_df["stability_pct"] >= 90).sum())
n_fragile = int((stability_df["stability_pct"] < 60).sum())
n_mid = int(((stability_df["stability_pct"] >= 60) & (stability_df["stability_pct"] < 90)).sum())
print(f"\n  Robustas    (stability ≥ 90%): {n_robust:>2}/{len(stability_df)}")
print(f"  Intermedias (60-89%)          : {n_mid:>2}/{len(stability_df)}")
print(f"  Frágiles    (< 60%)           : {n_fragile:>2}/{len(stability_df)}")
print(f"  Holm-sig vs azar (1/{k_anc}):  {n_sig_boot:>2}/{len(stability_df)} "
      f"(rechaza H0 tras corrección múltiple)")

weak = stability_df[stability_df["stability_pct"] < 80]
if len(weak) > 0:
    print("\n  Asignaciones con stability < 80%:")
    print(f"  {'variedad':20s}    {'canónica':12s}  {'stab':>7s}  {'top alt':12s}  {'%':>6s}")
    print("  " + "─" * 68)
    for _, r in weak.iterrows():
        print(f"  {r['variedad']:20s} →  {r['ancla_canonica']:12s} {r['stability_pct']:6.1f}%  "
              f"{r['top_alternative']:12s} {r['top_alt_pct']:5.1f}%")


# ── 5.1.2  Silhouette sobre TODAS las observaciones ─────────────────
print("\n" + "=" * 70)
print("Silhouette sobre todas las observaciones (no solo centroides)")
print("=" * 70)

sil_all, X_all, y_all = silhouette_over_observations(scaled_data, anchor_of_map, config.features)
sil_centroides = float(validation_tests[0].get("score", 0.0))
qual_all = "Excelente" if sil_all > 0.5 else "Aceptable" if sil_all > 0.25 else "Bajo"
print(f"\n  Observaciones usadas: {len(X_all):,}")
print(f"  Silhouette global:    {sil_all:+.4f}")
print(f"  (vs. {sil_centroides:+.4f} sobre centroides)")
print(f"  Calidad: {qual_all}")


# ── 5.1.3  Mann-Whitney U con corrección Holm-Bonferroni ────────────
print("\n" + "=" * 70)
print(f"Mann-Whitney U con corrección Holm-Bonferroni (α={config.alpha})")
print("=" * 70)

mw_holm_df, mw_holm_threshold_calibrated = mann_whitney_holm(
    validation_df, all_data, anchor_names, config.features,
    alpha=config.alpha, null_percentile=config.mw_holm_null_percentile,
    fallback_pct=config.min_similar_pct,
)
print(f"  Umbral MW-Holm calibrado: p{config.mw_holm_null_percentile:.0f} del null = "
      f"{mw_holm_threshold_calibrated:.1f}%  (fijo anterior: {config.min_similar_pct:.0f}%)")

n_raw = int((validation_df["pct_similar"] >= config.min_similar_pct).sum())
n_holm = int((mw_holm_df["pct_similar_holm"] >= config.min_similar_pct).sum())
n_holm_cal = int((mw_holm_df["pct_similar_holm"] >= mw_holm_threshold_calibrated).sum())
delta = n_raw - n_holm
print(f"\n  Asignaciones ≥{config.min_similar_pct:.0f}% (raw MW):         {n_raw:>2}/{len(validation_df)}")
print(f"  Asignaciones ≥{config.min_similar_pct:.0f}% (Holm, fijo):      {n_holm:>2}/{len(mw_holm_df)}")
print(f"  Asignaciones ≥{mw_holm_threshold_calibrated:.1f}% (Holm, calibrado): {n_holm_cal:>2}/{len(mw_holm_df)}")
if delta > 0:
    print(f"  → {delta} asignaciones bajaron al corregir por comparaciones múltiples")
elif delta < 0:
    print(f"  → {abs(delta)} asignaciones subieron (raro — revisar)")
else:
    print("  → Sin cambios tras la corrección")


# ── 5.1.4  Resumen ejecutivo del análisis reforzado ─────────────────
print("\n" + "=" * 70)
print("RESUMEN — Análisis estadístico reforzado")
print("=" * 70)
print(f"  Bootstrap:  {n_robust}/{len(stability_df)} asignaciones robustas (≥90%)")
print(f"  Bootstrap Holm-sig (vs 1/{k_anc}): {n_sig_boot}/{len(stability_df)}")
print(f"  Silhouette: {sil_all:+.4f} sobre {len(X_all):,} obs — {qual_all}")
print(f"  MW Holm:    {n_holm_cal}/{len(mw_holm_df)} asignaciones "
      f"≥{mw_holm_threshold_calibrated:.1f}% (umbral calibrado)")
print("\n✓ stability_df, sil_all, mw_holm_df, mw_holm_threshold_calibrated listos para la sección 6")


Bootstrap stability (100 iter por variedad no-ancla)



  Robustas    (stability ≥ 90%):  2/12
  Intermedias (60-89%)          :  1/12
  Frágiles    (< 60%)           :  9/12
  Holm-sig vs azar (1/11):   6/12 (rechaza H0 tras corrección múltiple)

  Asignaciones con stability < 80%:
  variedad                canónica         stab  top alt            %
  ────────────────────────────────────────────────────────────────────
  MALIBU               →  VENTURA         0.0%  JUPITER      100.0%
  BILOXI               →  VENTURA         0.0%  JUPITER      100.0%
  STELLA               →  ATLAS           0.0%  JUPITER      100.0%
  TERRAPIN             →  BIANCA          1.0%  JUPITER       79.0%
  FCM17-132            →  POP             2.0%  BEAUTY        76.0%
  MASIRAH              →  ATLAS          17.0%  BIANCA        49.0%
  AZRA                 →  BELLA          18.0%  ROSITA        68.0%
  MAGNUS               →  ROSITA         42.0%  POP           29.0%
  MADEIRA              →  VENTURA        56.0%  BIANCA        44.0%

Silhouette sobre 


  Observaciones usadas: 16,047
  Silhouette global:    -0.1120
  (vs. -0.0440 sobre centroides)
  Calidad: Bajo

Mann-Whitney U con corrección Holm-Bonferroni (α=0.05)


  Umbral MW-Holm calibrado: p90 del null = 40.0%  (fijo anterior: 60%)

  Asignaciones ≥60% (raw MW):          2/12
  Asignaciones ≥60% (Holm, fijo):       4/12
  Asignaciones ≥40.0% (Holm, calibrado):  7/12
  → 2 asignaciones subieron (raro — revisar)

RESUMEN — Análisis estadístico reforzado
  Bootstrap:  2/12 asignaciones robustas (≥90%)
  Bootstrap Holm-sig (vs 1/11): 6/12
  Silhouette: -0.1120 sobre 16,047 obs — Bajo
  MW Holm:    7/12 asignaciones ≥40.0% (umbral calibrado)

✓ stability_df, sil_all, mw_holm_df, mw_holm_threshold_calibrated listos para la sección 6


---
## 6. Tablero de ruteo a anclas (decisión por error predictivo)

Acá está **la decisión accionable**. No es distribucional ni clustering: cada variedad
se rutea al ancla cuyo **modelo la pronostica mejor** (MAPE out-of-sample). Las
métricas de las secciones 2-5 (efecto, estabilidad, silhouette) quedan como
**diagnóstico de soporte**.

| Decisión | Criterio | Acción |
|---|---|---|
| 🔵 **ancla** | una de las 11 grandes | entrena modelo propio |
| 🟢 **bueno** | MAPE ≤ 1.5× el típico de un ancla | hereda con confianza |
| 🟢 **aceptable** | ≤ 2.0× | hereda, monitorear |
| 🟡 **revisar** | > 2.0× | distinta + poca data → mejores features / más datos |

### Cómo leer dos columnas que pueden confundir

- **`ganancia` negativa (▼):** la variedad pronostica **mejor con su propio modelo**
  que heredando. Si además tiene `n` holgado (cientos de filas), es **candidata a
  ancla**, no a ruteo (histórico: BELLA, ARANA, RAYMI). Revisar antes de congelar el
  mapping.
- **ATLAS — ancla atípica:** se autopronostica con MAPE muy alto (~75% en el proxy);
  el modelo proxy no la captura. Eso **infla el baseline** y abarata los `ratio` del
  resto → las categorías quedan **optimistas**. Tratarla aparte (ver
  [SUSTENTO_ESTADISTICO_RUTEO.md](SUSTENTO_ESTADISTICO_RUTEO.md) §7.1).

> **Salida:** `decision_table.html` (tablero) + `variety_predictive_routing.yaml`
> (mapping `variedad → ancla` para el pipeline). El MAPE es **proxy** (sin lag features
> del pipeline real); fija el **ruteo**, no el MAPE de producción.

In [11]:
# ══════════════════════════════════════════════════════════════════
# 6.  Tablero de RUTEO A ANCLAS (decisión por error predictivo)
# ------------------------------------------------------------------
# La lógica vive en variety_routing.route_to_anchors_predictive (transferencia,
# no clustering). Aquí solo: render del tablero + export de artefactos.
# Columna clave nueva: GANANCIA = cuánto baja el error al heredar vs modelo propio.
# ══════════════════════════════════════════════════════════════════
from datetime import datetime
from pathlib import Path

import yaml

TARGET = "KGHORA"  # = KG/JR_H, target real
PREDICTORS = ["KGHECT", "INDUSTRIAL", "DPC", "PesoBayaFIMPRO"]  # nunca el target (leakage)
# Anclas: ÚNICA fuente de verdad → config.anchor_varieties (variety_routing.py).
# §1-§5 (prior distribucional) y §7 (decisión predictiva) usan el MISMO set
# → el diagnóstico de soporte describe exactamente la partición que se decide.
ROUTING_ANCHORS = [a for a in config.anchor_varieties if a in all_data]

routing_df, baseline = route_to_anchors_predictive(
    all_data, ROUTING_ANCHORS, PREDICTORS, TARGET
)

# Diagnósticos distribucionales de soporte (secciones 5.1)
stab_map = dict(zip(stability_df["variedad"], stability_df["stability_pct"]))
eff_map = dict(zip(effect_df["variedad"], effect_df["max_cliff"]))
COLOR = {"ancla": "#0ea5e9", "bueno": "#059669", "aceptable": "#84cc16", "revisar": "#f59e0b"}

# ── Export artefactos ──
out_dir = Path(config.output_dir)
out_dir.mkdir(parents=True, exist_ok=True)
routing_df.round(2).to_csv(out_dir / "decision_table.csv", index=False)
mapping = dict(zip(routing_df["variedad"], routing_df["entrena_con"]))
with open(out_dir / "variety_predictive_routing.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump({"anchors": ROUTING_ANCHORS, "routing": mapping}, f,
                   allow_unicode=True, sort_keys=False)
routing_df[["variedad", "entrena_con", "decision", "mape_oos", "ganancia"]].round(2).to_csv(
    out_dir / "variety_predictive_routing.csv", index=False)

# ── Resumen en stdout ──
n_anchor = int((routing_df.decision == "ancla").sum())
n_good = int(routing_df.decision.isin(["ancla", "bueno", "aceptable"]).sum())
n_review = int((routing_df.decision == "revisar").sum())
routed = routing_df[routing_df.decision != "ancla"]
gan_med = routed["ganancia"].mean()
print("=" * 74)
print(f"RUTEO A ANCLAS — {n_anchor} anclas · {len(routing_df) - n_anchor} ruteadas")
print("=" * 74)
print(routing_df[["variedad", "entrena_con", "n", "mape_oos", "mape_propio", "ganancia", "decision"]]
      .round(1).to_string(index=False))
print(f"\n  Confiables (ancla+bueno+aceptable): {n_good}/{len(routing_df)}  ·  A revisar: {n_review}")
print(f"  Ganancia media al heredar (ruteadas): {gan_med:+.1f} pp de MAPE")
print(f"  MAPE típico de un ancla (proxy): {baseline:.1f}%")

# ── HTML ──
def _gan_cell(r):
    if r["decision"] == "ancla":
        return "<small>—</small>"
    g = r["ganancia"]
    if pd.isna(g):
        return "<small>sin data propia</small>"
    if g > 0:
        return f"<span style='color:#059669;font-weight:600'>▲ {g:.1f} pp</span>"
    return f"<span style='color:#dc2626;font-weight:600'>▼ {abs(g):.1f} pp</span>"

now = datetime.now().strftime("%Y-%m-%d %H:%M")
kpis = [
    ("Variedades", f"{len(routing_df)}"),
    ("Anclas (modelo propio)", f"{n_anchor}"),
    ("Ruteadas", f"{len(routing_df) - n_anchor}"),
    ("Confiables", f"{n_good}/{len(routing_df)}"),
    ("A revisar", f"{n_review}"),
    ("Ganancia media al heredar", f"{gan_med:+.1f} pp"),
    ("MAPE típico ancla (proxy)", f"{baseline:.1f}%"),
    ("Estructura (diagnóstico)", "sin clusters" if sil_all <= 0.25 else "clusters"),
]
kpi_html = "".join(
    f"<div class='kpi'><div class='kpi-v'>{v}</div><div class='kpi-l'>{l}</div></div>"
    for l, v in kpis
)
rows_html = []
for _, r in routing_df.iterrows():
    ratio = "—" if r["decision"] == "ancla" else f"{r['ratio']:.2f}×"
    own = "—" if r["decision"] == "ancla" else (f"{r['mape_propio']:.1f}%" if pd.notna(r["mape_propio"]) else "—")
    cliff = f"{eff_map[r['variedad']]:.2f}" if r["variedad"] in eff_map and pd.notna(eff_map[r["variedad"]]) else "—"
    rows_html.append(
        f"<tr><td><code>{r['variedad']}</code></td><td><code>{r['entrena_con']}</code></td>"
        f"<td>{int(r['n'])}</td><td>{r['mape_oos']:.1f}%</td><td>{own}</td>"
        f"<td>{_gan_cell(r)}</td><td>{ratio}</td><td><small>{cliff}</small></td>"
        f"<td><span class='pill' style='background:{COLOR[r['decision']]}'>{r['decision']}</span></td></tr>"
    )

html_doc = f"""<!DOCTYPE html><html lang="es"><head><meta charset="utf-8">
<title>Ruteo a anclas — {now}</title><style>
 * {{ box-sizing:border-box }} body {{ font-family:-apple-system,Segoe UI,Roboto,sans-serif; margin:0; background:#f8fafc; color:#0f172a }}
 .wrap {{ max-width:1200px; margin:0 auto; padding:32px 24px }}
 h1 {{ margin:0 0 4px; font-size:25px }} .sub {{ color:#64748b; margin-bottom:22px; font-size:13px }}
 .kpis {{ display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:12px; margin-bottom:24px }}
 .kpi {{ background:white; border:1px solid #e2e8f0; border-radius:10px; padding:16px; text-align:center }}
 .kpi-v {{ font-size:20px; font-weight:700 }} .kpi-l {{ font-size:11px; color:#64748b; text-transform:uppercase; letter-spacing:.04em; margin-top:4px }}
 .card {{ background:white; border:1px solid #e2e8f0; border-radius:10px; padding:16px; margin-bottom:22px }}
 h2 {{ font-size:17px; margin:0 0 10px }} table {{ width:100%; border-collapse:collapse; font-size:14px }}
 th,td {{ text-align:left; padding:8px 10px; border-bottom:1px solid #e2e8f0 }}
 th {{ background:#f1f5f9; font-size:12px; text-transform:uppercase; letter-spacing:.03em }}
 .pill {{ color:white; padding:2px 10px; border-radius:999px; font-size:12px; font-weight:600; text-transform:capitalize }}
 code {{ background:#f1f5f9; padding:1px 6px; border-radius:4px; font-size:12px }} small {{ color:#64748b }}
 .foot {{ color:#64748b; font-size:12px; margin-top:18px; line-height:1.7 }}
</style></head><body><div class="wrap">
 <h1>Decisión de ruteo a anclas (donante)</h1>
 <div class="sub">Generado el {now} · <b>metodología: transferencia / ruteo a donante</b> (no clustering) ·
  decisión por <b>MAPE out-of-sample</b> · {len(ROUTING_ANCHORS)} anclas · features proxy (sin lags)</div>
 <div class="kpis">{kpi_html}</div>
 <div class="card">
  <h2>Ruteo variedad → ancla (decisión por error de pronóstico)</h2>
  <p style="color:#64748b;font-size:13px;margin:0 0 10px">
   <b>MAPE OOS</b>: error del modelo del ancla sobre la variedad · <b>MAPE propio</b>: error de su modelo
   individual (CV) · <b>Ganancia</b>: puntos de MAPE que se ahorran al heredar (▲ = heredar mejora) ·
   <b>×baseline</b>: veces el MAPE típico de un ancla ({baseline:.1f}%) · <b>|Cliff|</b>: diagnóstico distribucional</p>
  <table><thead><tr><th>Variedad</th><th>Entrena con</th><th>n</th><th>MAPE OOS</th>
   <th>MAPE propio</th><th>Ganancia</th><th>×baseline</th><th>|Cliff|</th><th>Decisión</th></tr></thead>
   <tbody>{''.join(rows_html)}</tbody></table>
 </div>
 <div class="foot">
  <b>Qué se gana (columna Ganancia):</b> cuánto baja el error al usar el modelo del ancla en vez del
  modelo propio de la variedad. Las chicas casi siempre ganan porque su data sola no alcanza para un
  modelo bueno.<br><br>
  <b>Leyenda:</b>
  <span class='pill' style='background:#0ea5e9'>ancla</span> modelo propio ·
  <span class='pill' style='background:#059669'>bueno</span> MAPE ≤ 1.5× baseline ·
  <span class='pill' style='background:#84cc16'>aceptable</span> ≤ 2.0× ·
  <span class='pill' style='background:#f59e0b'>revisar</span> &gt; 2.0× (distinta + poca data → mejores features / más datos)
  <br><br><b>Estructura:</b> silhouette {sil_all:+.3f} sobre {len(X_all):,} obs → no hay clusters naturales
  (esperado: es ruteo a donante, no clustering). MAPE proxy (sin lag features); con el pipeline real baja.
 </div>
</div></body></html>"""
(out_dir / "decision_table.html").write_text(html_doc, encoding="utf-8")
print(f"\n✓ {out_dir}/decision_table.csv / .html")
print(f"✓ {out_dir}/variety_predictive_routing.csv / .yaml  (mapping accionable)")


RUTEO A ANCLAS — 11 anclas · 12 ruteadas
 variedad entrena_con    n  mape_oos  mape_propio  ganancia  decision
  MASIRAH     VENTURA   56      24.6         32.6       7.9 aceptable
    BELLA     JUPITER  195      24.7         21.6      -3.1 aceptable
  MADEIRA      BEAUTY   66      25.2         27.0       1.8 aceptable
    RAYMI         POP  258      25.3         22.8      -2.5 aceptable
FCM17-132       KIRRA   48      26.0         35.2       9.2 aceptable
     AZRA         POP   78      26.0         33.6       7.6 aceptable
    ARANA     JUPITER  311      27.8         27.0      -0.8 aceptable
   BONITA      BILOXI   39      28.1         31.9       3.8 aceptable
   BILOXI      BILOXI  180       4.6          4.6       0.0     ancla
   MAGICA      MAGICA  509       5.7          5.7       0.0     ancla
    KIRRA       KIRRA  375       6.3          6.3       0.0     ancla
   ROSITA      ROSITA  305       7.3          7.3       0.0     ancla
  JUPITER     JUPITER  662       7.4          7.4